# Chapter 1: Your First Policy
### *Expected Value*

Welcome to **Arclight Cyber Insurance**.

You've been hired as the company's first actuary. Your CEO has landed the company's very first prospective policyholder — **Meridian Medical Group**, a regional healthcare provider with \$10 million in annual revenue.

They want a standalone cyber liability policy: first-party coverage for their own losses (ransomware recovery, breach response, business interruption) and third-party liability for patient data lawsuits.

**Your job: set the annual premium.**

Price it right, and the company profits. Price it too low, and one ransomware incident wipes you out. Price it too high, and Meridian goes to a competitor.

## The math

For each threat vector, the **expected annual loss** is the frequency (probability of at least one incident per year) times the severity (fraction of annual revenue consumed by the incident) times annual revenue:

$$ \text{expected loss}_i = f_i \cdot s_i \cdot R $$

The **pure premium** is the sum across all threat vectors — the minimum needed to cover expected claims:

$$ P_{\text{pure}} = \sum_i f_i \cdot s_i \cdot R $$

The **gross premium** grosses up for expenses (claims handling, incident response retainers, overhead, taxes), charged as a fraction $e$ of premium:

$$ P_{\text{gross}} = \frac{P_{\text{pure}}}{1 - e} $$

Anything above $P_{\text{gross}}$ is profit margin.

> **Why use revenue?** Cyber losses scale with the size of the business — a ransomware attack on a 10M company costs roughly 10× more to recover from than one on a \$1M company. Revenue is the standard exposure base in cyber ratemaking.

In [9]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

## The policy

Meridian Medical Group has \$10M in annual revenue. Three threat vectors drive their cyber exposure. Expense ratio is 35% — cyber lines carry higher operating costs than traditional P&C due to incident response retainers and specialized claims handling.

In [10]:
@dataclass
class ThreatVector:
    name: str
    frequency: float   # probability of at least one incident per year
    severity_pct: float  # fraction of annual revenue consumed by a single incident

THREATS = [
    ThreatVector("Ransomware",                  0.08, 0.0250),
    ThreatVector("Data Breach",                 0.04, 0.0200),
    ThreatVector("Business Email Compromise",   0.10, 0.0040),
]

ANNUAL_REVENUE  = 10_000_000
EXPENSE_RATIO   = 0.35
MAX_MARKET_PREMIUM = 150_000

pd.DataFrame([vars(t) for t in THREATS])

,name,frequency,severity_pct
0,Ransomware,0.08,0.025
1,Data Breach,0.04,0.020
2,Business Email Compromise,0.10,0.004


## Calculate the pure and gross premium

Before you price the policy, work out the floor.

In [11]:
def expected_pure_premium(threats, revenue):
    return sum(t.frequency * t.severity_pct * revenue for t in threats)

def expected_gross_premium(threats, revenue, expense_ratio):
    return expected_pure_premium(threats, revenue) / (1 - expense_ratio)

PURE  = expected_pure_premium(THREATS, ANNUAL_REVENUE)
GROSS = expected_gross_premium(THREATS, ANNUAL_REVENUE, EXPENSE_RATIO)

print(f"Pure premium:  ${PURE:,.2f}   (covers expected claims only)")
print(f"Gross premium: ${GROSS:,.2f}   (covers claims + {EXPENSE_RATIO:.0%} expenses)")

Pure premium:  $32,000.00   (covers expected claims only)
Gross premium: $49,230.77   (covers claims + 35% expenses)


## Simulate a year

Each threat vector rolls independently. If it fires, the claim is `severity_pct × annual_revenue`.

Note: in real cyber, multiple incidents can compound (a breach can lead to ransomware). That's a Chapter 3 problem.

In [12]:
def simulate_year(threats, revenue, premium, expense_ratio, rng):
    claims = [
        {"vector": t.name, "amount": revenue * t.severity_pct}
        for t in threats if rng.random() < t.frequency
    ]
    claims_incurred = sum(c["amount"] for c in claims)
    expenses = premium * expense_ratio
    underwriting_income = premium - claims_incurred - expenses
    loss_ratio     = claims_incurred / premium if premium > 0 else 0.0
    combined_ratio = (claims_incurred + expenses) / premium if premium > 0 else 0.0
    return {
        "premium": premium,
        "num_claims": len(claims),
        "claims_incurred": claims_incurred,
        "expenses": expenses,
        "underwriting_income": underwriting_income,
        "loss_ratio": loss_ratio,
        "combined_ratio": combined_ratio,
    }

## Your turn: set the premium

Pick a premium, pick a run length, hit **Run**. The charts show per-year underwriting income and cumulative surplus. With only one policy, expect high year-to-year variance — a single ransomware year can swing the whole picture. That's Chapter 2's problem.

In [13]:
def evaluate(premium, results):
    total_premiums = sum(r["premium"] for r in results)
    total_claims   = sum(r["claims_incurred"] for r in results)
    total_expenses = sum(r["expenses"] for r in results)
    total_income   = total_premiums - total_claims - total_expenses
    avg_loss_ratio = np.mean([r["loss_ratio"] for r in results])
    cum = np.cumsum([r["underwriting_income"] for r in results])
    went_insolvent = bool((cum < 0).any())

    lines = []
    lines.append(f"<p>Over <b>{len(results)}</b> years: "
                 f"earned ${total_premiums:,.0f} in premiums, "
                 f"paid ${total_claims:,.0f} in claims, "
                 f"${total_expenses:,.0f} in expenses.</p>")
    lines.append(f"<p>Net underwriting income: <b>${total_income:,.0f}</b>. "
                 f"Average loss ratio: <b>{avg_loss_ratio:.1%}</b>.</p>")

    if premium < PURE:
        lines.append(f"<p style='color:crimson'><b>FAIL.</b> Premium (${premium:,.0f}) is below the pure premium (${PURE:,.0f}). You didn't even cover expected losses.</p>")
    elif premium < GROSS:
        lines.append(f"<p style='color:darkorange'><b>UNDERPRICED.</b> Premium (${premium:,.0f}) covered losses but not expenses. Break-even is ${GROSS:,.0f}.</p>")
    elif premium > MAX_MARKET_PREMIUM:
        lines.append(f"<p style='color:darkorange'><b>OVERPRICED.</b> ${premium:,.0f} is above the market rate of ${MAX_MARKET_PREMIUM:,.0f}. Meridian goes to a competitor.</p>")
    else:
        margin = (premium - GROSS) / premium * 100
        lines.append(f"<p style='color:seagreen'><b>PASS.</b> ${premium:,.0f} covers expected losses (${PURE:,.0f}), expenses, and includes a {margin:.1f}% profit margin.</p>")

    if went_insolvent:
        lines.append("<p style='color:crimson'>Company went insolvent mid-simulation. With one policy, a single ransomware year can sink you — a lesson for Chapter 2.</p>")

    lines.append("<hr><p><b>Key takeaway.</b> The pure premium is the floor — the minimum to cover expected claims. The gross premium adds expense loading. Any margin above that is profit. In cyber, this floor is harder to estimate than in property: the data is sparse, threats evolve fast, and a single tail event can dwarf years of premiums.</p>")
    display(HTML(''.join(lines).replace('$', '&#36;')))


premium_slider   = widgets.FloatSlider(value=GROSS * 1.1, min=10_000, max=200_000, step=1_000,
                                       description="Premium:", readout_format=",.0f",
                                       layout=widgets.Layout(width="500px"))
num_years_slider = widgets.IntSlider(value=10, min=1, max=100, description="Years:",
                                     layout=widgets.Layout(width="500px"))
run_button       = widgets.Button(description="Run simulation", button_style="primary")
output           = widgets.Output()

def run_sim(_):
    premium = premium_slider.value
    n_years = num_years_slider.value
    rng = np.random.default_rng()
    results = [simulate_year(THREATS, ANNUAL_REVENUE, premium, EXPENSE_RATIO, rng)
               for _ in range(n_years)]
    df = pd.DataFrame(results)
    df["year"] = df.index + 1
    df["cumulative_income"] = df["underwriting_income"].cumsum()

    with output:
        output.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        colors = np.where(df["underwriting_income"] >= 0, "seagreen", "crimson")
        axes[0].bar(df["year"], df["underwriting_income"], color=colors)
        axes[0].axhline(0, color="black", linewidth=0.5)
        axes[0].set_title("Underwriting income by year")
        axes[0].set_xlabel("Year")
        axes[0].set_ylabel("$")
        axes[1].plot(df["year"], df["cumulative_income"], marker="o")
        axes[1].axhline(0, color="black", linewidth=0.5)
        axes[1].set_title("Cumulative surplus")
        axes[1].set_xlabel("Year")
        axes[1].set_ylabel("$")
        plt.tight_layout()
        plt.show()

        display(df[["year", "num_claims", "claims_incurred", "expenses",
                    "underwriting_income", "loss_ratio", "combined_ratio",
                    "cumulative_income"]].style.format({
            "claims_incurred":     "${:,.0f}",
            "expenses":            "${:,.0f}",
            "underwriting_income": "${:,.0f}",
            "loss_ratio":          "{:.1%}",
            "combined_ratio":      "{:.1%}",
            "cumulative_income":   "${:,.0f}",
        }))
        evaluate(premium, results)

run_button.on_click(run_sim)
display(widgets.VBox([premium_slider, num_years_slider, run_button, output]))

## Hints

<details><summary>Hint 1 — where to start</summary>

Think about the expected cost of each threat vector separately. What's the average annual loss from ransomware hitting Meridian?

</details>

<details><summary>Hint 2 — the formula</summary>

Expected loss per vector = frequency × severity fraction × annual revenue. Calculate this for each vector and add them up.

</details>

<details><summary>Hint 3 — the numbers</summary>

```
Ransomware               = 0.08 × 0.0250 × \$10,000,000 = \$20,000
Data Breach              = 0.04 × 0.0200 × \$10,000,000 =  \$8,000
Business Email Compromise= 0.10 × 0.0040 × \$10,000,000 =  \$4,000
Pure premium total                                      = \$32,000
```

</details>

<details><summary>Hint 4 — expense loading</summary>

The pure premium only covers expected claims. 35% of premium goes to expenses (incident response retainer, claims handlers, overhead), so:

```
gross premium = pure_premium / (1 − 0.35)
              = \$32,000 / 0.65
              = \$49,231
```

Add a profit margin on top.

</details>

## What's next

**Chapter 2 — Growing the Book (Law of Large Numbers).** You priced one policy well, but one policy is a coin flip. In Chapter 2 you'll write thousands of policies and watch how volume tames that per-policy volatility.

There's a catch, though. In property insurance, a thousand homes don't all burn down at once. In cyber, a single piece of malware — WannaCry, NotPetya — can simultaneously detonate across your entire book of business. The law of large numbers assumes independence. Chapter 2 will show you what happens when it breaks.